In [6]:
import numpy as np
import pandas as pd

# 1. Leer el archivo bruto de ocupación del INE
df_ocupacion = pd.read_csv(
    "ine_grado_ocupacion_bruto.csv",
    encoding="latin1",
    encoding_errors="ignore",
    sep=";",
)

if "Periodo" not in df_ocupacion.columns and len(df_ocupacion.columns) == 1:
    df_ocupacion = pd.read_csv(
        "ine_grado_ocupacion_bruto.csv",
        encoding="latin1",
        encoding_errors="ignore",
        sep=",",
    )

# 2. Limpieza estricta de números (comas por puntos) y fechas
df_ocupacion["Total"] = (
    df_ocupacion["Total"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df_ocupacion["Total"] = df_ocupacion["Total"].replace(["", "."], np.nan)
df_ocupacion["Total"] = df_ocupacion["Total"].astype(float)
df_ocupacion["Periodo"] = (
    df_ocupacion["Periodo"].astype(str).str.replace("M", "-")
)

# 3. Nos quedamos SOLO con Periodo y cambiamos 'Total' por 'grado_ocupacion'
df_ocupacion = df_ocupacion[["Periodo", "Total"]].rename(
    columns={"Total": "grado_ocupacion"}
)

# 4. Recuperar el dataset de viajeros base (eliminando duplicados previos)
df_existente = pd.read_csv("ine_viajeros_clean.csv")
if "grado_ocupacion" in df_existente.columns:
    df_existente = df_existente.drop(columns=["grado_ocupacion"])
columnas_a_borrar = [
    c for c in df_existente.columns if c.startswith("grado_ocupacion_")
]
if columnas_a_borrar:
    df_existente = df_existente.drop(columns=columnas_a_borrar)

# 5. Fusionar ambos datasets en una sola columna limpia
df_final = pd.merge(df_existente, df_ocupacion, on="Periodo", how="left")

# 6. Orden descendente (2025 arriba)
df_final = df_final.sort_values(by="Periodo", ascending=False)

# 7. REORDENAR COLUMNAS SEGÚN TU ESPECIFICACIÓN EXACTA
columnas_ordenadas = [
    "Periodo",
    "pernoctaciones_totales",
    "viajeros_totales",
    "grado_ocupacion",
    "viajeros_nacionales",
    "viajeros_internacionales",
    "pernoctaciones_nacionales",
    "pernoctaciones_internacionales",
]
df_final = df_final[columnas_ordenadas]

# 8. Guardar en el CSV definitivo
df_final.to_csv("ine_viajeros_clean.csv", index=False)

# 9. Mostrar resultado final en Jupyter
df_final.head()

,Periodo,pernoctaciones_totales,viajeros_totales,grado_ocupacion,viajeros_nacionales,viajeros_internacionales,pernoctaciones_nacionales,pernoctaciones_internacionales
0,2025-12,522493.0,247182.0,61.77,111847.0,135335.0,199937.0,322556.0
1,2025-11,560063.0,259890.0,68.03,107220.0,152670.0,190555.0,369508.0
2,2025-10,628123.0,288525.0,75.44,87542.0,200983.0,148530.0,479593.0
3,2025-09,571732.0,280438.0,70.57,96557.0,183881.0,158434.0,413298.0
4,2025-08,523211.0,250889.0,62.09,84305.0,166584.0,143076.0,380135.0
